# Tiny Recursive Model & Parallel scaling

TL;DR: The project is about latent reasoning, practically making Tiny Recursive Model (TRM) work better.

**Literature**:
 - Practice: https://www.arxiv.org/pdf/2510.04871, https://arcprize.org/blog/hrm-analysis
 - Theory: https://arxiv.org/pdf/2505.12514v3, https://arxiv.org/pdf/2510.15522

**Intuition**: For tasks like Sudoku, the search algorithms typically do DFS or BFS with constraint propagation and backtracking.

While learning constraint propagation (i.e. filling in the last cell of the row / column / box) is not hard, assigning a value and then backtracking may be not that easy.

First thing that comes to mind is to use layers as actions and somehow evaluate the latent state $z$ (e.g. by the entropy of the predicted distribution) to run the DFS optimization in the discrete space.

However, the latents can encode not only one answer, but a **superposition** of multiple answers. For example, let $z$ be the latent, and $e_i$ be the latent vector that encodes a particular solution $S_i$, i.e. $D(e_i) \sim [\mathbb{P}(S = S_i) = 1]$.

Then, we can write $z$ as a linear combination of the $e_i$:

$$z = \sum_i \alpha_i e_i$$

What is nice is that to arrive to the correct solution $e_c$, we just need to amplify the correct $\alpha_c$ and suppress the rest $\alpha_i \to 0, i \neq c$. (For quantum explanation, see [this](https://youtu.be/RQWpF2Gb-gU?si=FU-dJHSF_sGt75-r)). Since it is a linear space, we can add or subtract latents from each other.

We hypothesize we start with $\alpha_i = \frac{1}{N}$, and at each step of the recursive update ($z_{t+1} \to f(z_t, \dots)$), some of the incorrect solutions are suppressed ($\alpha_i \approx 0$), in other words $\alpha_i$ becomes more concentrated. This is nice for resolving constraints.

However, for search with backtracking, we have to assume some value of the cell, essentially zeroing out some of the solutions. If the assumption was wrong, when we backtrack, we have to recover that zeroed out vector, and REMEMBER not to do that again. If we only operate on one latent z_t, this is not possible since we will arrive to the same value and end in the loop.

With two latents, we can do the following:

$$z_t' = z_t - \sum_{i \in C} \alpha_i e_i$$

where $z_t'$ is the new latent, $C$ is the set of solutions that do not satisfy the constraint, and $\alpha_i$ is the weight of the latent vector $e_i$.
Then we run $f(z_t', \dots)$ and get $z_{t+1} = z_t + z_t'$. If the assumption was wrong, the $f(z_t', \dots)$ should just be equal to zero (or something such that we do not arrive to the same $z_t'$ again).

**Idea**: To evaluate multiple candidates for the solution, at each high level step, let's generate a partition of the $z$ vector:

1. Split the latent $z$ into $K$ parts: $z \to g(x, y, z) = \{z_i\}^{K}_{i=1}$, s.t. $z = \sum_{i=1}^{K} z_i$
2. In parallel, calculate the $f(z_i)$ (recursively), and update the latents: $z \to \sum_{i=1}^{K} f(x, y, z_i)$

where $z_i$ is the latent vector that encodes a particular set of solutions $S_i$. We can check that $H(D(z_i)) < H(D(z))$ for all $1 \leq i \leq K$.